In [2]:
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
plt.rcParams['font.family']='Times New Roman,Microsoft YaHei'# 设置字体族，中文为微软雅黑，英文为Times New Roman
plt.rcParams['mathtext.fontset'] = 'stix' # 设置数%matplotlib qt学公式字体为stix
plt.style.use('seaborn-v0_8-paper')
# 设置全局参数
plt.rcParams['figure.facecolor'] = 'white'  # 设置图形的背景为透明
plt.rcParams['axes.facecolor'] = 'white'    # 设置轴域的背景为透明
plt.rcParams['savefig.facecolor'] = 'white' # 保存图像时背景透明
import matplotlib
# matplotlib.use('TkAgg')
%matplotlib inline

In [3]:
df_chordkey=pd.read_pickle("df_chordkey,pkl")

In [8]:
from sklearn.preprocessing import LabelEncoder
from hmmlearn import hmm,vhmm
import pickle
import os
from tqdm import tqdm
os.environ['OMP_NUM_THREADS'] = '10'  # 例如，设置为4个线程
# 准备存储转移矩阵的字典
transition_matrices = {}
# 获取所有唯一的 'place'，'level' 和 'Month'
places = df_chordkey['place'].unique()

In [6]:
def removechordkey_month(df_chordkey,chordbound=0.05,keybound=0.05):
    df_chordkey['Month'] = df_chordkey['Date'].dt.month
    month_order = [12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    df_chordkey['Month'] = pd.Categorical(df_chordkey['Month'], categories=month_order, ordered=True)
    df_chordkey = df_chordkey.sort_values('Month')
    month_labels = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
                    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
    df_chordkey['Monthtext'] = df_chordkey['Month'].map(month_labels)
    month_order = ['Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov']
    df_chordkey['Monthtext'] = pd.Categorical(df_chordkey['Monthtext'], categories=month_order, ordered=True)
    df_chordkey = df_chordkey.sort_values('Monthtext')
    df_chordkey['sample']=df_chordkey["Monthtext"]
    
    # 计算每个和弦的出现次数
    event_counts = df_chordkey['event'].value_counts()
    # 计算每个track的出现次数
    track_counts = df_chordkey['track'].value_counts()
    # 计算总数的1%
    event_threshold = len(df_chordkey) * chordbound
    track_threshold = len(df_chordkey) * keybound
    # 过滤掉出现次数低于1%的和弦和track
    filtered_events = event_counts[event_counts > event_threshold].index.tolist()
    filtered_tracks = track_counts[track_counts > track_threshold].index.tolist()
    # 创建一个过滤后的DataFrame
    filtered_df = df_chordkey[df_chordkey['event'].isin(filtered_events) & df_chordkey['track'].isin(filtered_tracks)]
    return filtered_df

In [7]:
# 计算总的迭代次数（总的组合数）
total_combinations = len(places) 

# 使用 tqdm 进度条包装循环
for place in tqdm(places, desc="Processing places"):
    # 过滤数据
    filtered_df = removechordkey_month(df_chordkey.copy(),chordbound=0.01,keybound=0.01)
    # 按 Date 排序
    filtered_df = filtered_df.sort_values(by='Date')
    # 将和弦（event）编码为整数
    le = LabelEncoder()
    encoded_events = le.fit_transform(filtered_df['event'])
    # 准备和弦序列数据
    sequences = encoded_events.reshape(-1, 1)
    lengths = len(sequences)
    if lengths > 1:  # 确保有足够的数据来训练 HMM
        # 创建并训练 HMM 模型
        n_states = len(le.classes_)  # 独特的和弦数量作为状态数
       # 定义隐马尔可夫模型，使用MultinomialHMM
        model = vhmm.VariationalCategoricalHMM(
            n_components=n_states,  # 状态数
            n_iter=1000            # 最大迭代次数
        )

        model.fit(sequences)

        # 获取转移矩阵
        transition_matrix = model.transmat_

        # 创建转移矩阵的 DataFrame，并将和弦名称作为索引和列
        transition_matrix_df = pd.DataFrame(
            transition_matrix, 
            index=le.classes_, 
            columns=le.classes_
        )

        # 构建字典的键
        key = f'{place}'
        transition_matrices[key] = transition_matrix_df

Processing places:   0%|          | 0/3 [54:00<?, ?it/s]


KeyboardInterrupt: 

In [9]:
# 过滤数据
filtered_df = removechordkey_month(df_chordkey.copy(),chordbound=0.01,keybound=0.01)
# 按 Date 排序
filtered_df = filtered_df.sort_values(by='Date')
# 将和弦（event）编码为整数
le = LabelEncoder()
encoded_events = le.fit_transform(filtered_df['event'])
# 准备和弦序列数据
sequences = encoded_events.reshape(-1, 1)
lengths = len(sequences)
if lengths > 1:  # 确保有足够的数据来训练 HMM
    # 创建并训练 HMM 模型
    n_states = len(le.classes_)  # 独特的和弦数量作为状态数
   # 定义隐马尔可夫模型，使用MultinomialHMM
    model = vhmm.VariationalCategoricalHMM(
        n_components=n_states,  # 状态数
        n_iter=1000            # 最大迭代次数
    )

    model.fit(sequences)

    # 获取转移矩阵
    transition_matrix = model.transmat_

    # 创建转移矩阵的 DataFrame，并将和弦名称作为索引和列
    transition_matrix_df = pd.DataFrame(
        transition_matrix, 
        index=le.classes_, 
        columns=le.classes_
    )

    # 构建字典的键
    key = f'all'
    transition_matrices[key] = transition_matrix_df

KeyboardInterrupt: 

In [10]:
filtered_df 

,sample,track,event,place,level,Date,Month,Monthtext
1048160,Oct,G major,G:maj,CM,1.5 m,2022-10-27 11:30:00,10,Oct
1045947,Oct,G minor,G:maj,CM,16 m,2022-10-27 11:40:00,10,Oct
1117091,Oct,G minor,G:maj,CM,10 m,2022-10-27 11:40:00,10,Oct
1048159,Oct,G major,G:maj,CM,1.5 m,2022-10-27 11:40:00,10,Oct
1037013,Oct,G minor,G:maj,CM,22 m,2022-10-27 11:40:45,10,Oct
...,...,...,...,...,...,...,...,...
915563,Jun,G minor,G#:maj,JH,4 m,2024-06-13 13:10:00,6,Jun
884426,Jun,A minor,A#:maj,JH,14 m,2024-06-13 13:10:00,6,Jun
884427,Jun,A minor,G#:maj,JH,14 m,2024-06-13 13:10:00,6,Jun
915568,Jun,D minor,F#:maj,JH,4 m,2024-06-13 13:20:00,6,Jun


In [ ]:
# 转移矩阵字典保存为文件（例如，pickle 文件）
with open('transition_matrices_all.pkl', 'wb') as f:
    pickle.dump(transition_matrices, f)